In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay,
)
from xgboost import XGBClassifier

SEED = 42
rng = np.random.default_rng(SEED)
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid")


# ==========================================================================
# STEP 1: GENERATE THE SYNTHETIC BEHAVIOURAL DATASET
# ==========================================================================
N = 3200

age   = rng.integers(18, 60, N)
income = rng.integers(15_000, 120_000, N)
employment = rng.choice(
    ["Student", "Salaried", "Self-employed", "Unemployed"],
    size=N, p=[0.20, 0.50, 0.20, 0.10])
spend = rng.integers(5_000, 90_000, N)
bnpl  = rng.integers(1, 25, N)
atv   = rng.integers(500, 10_000, N)
late  = rng.integers(0, 7, N)
credit = rng.integers(300, 850, N)
sti = spend / income                      # engineered: spending-to-income


def z(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std()


# Latent risk -> probability -> SAMPLED label.
# Coefficient signs encode domain logic; their sizes set feature importance.
logit = (
    -1.30
    + 1.40 * z(late)                       # more late payments  -> higher risk
    - 1.20 * z(credit)                     # lower credit score  -> higher risk
    + 0.80 * z(sti)                        # over-spending       -> higher risk
    + 0.40 * z(bnpl)                       # heavy BNPL use      -> higher risk
    - 0.30 * z(income)                     # higher income       -> lower risk
    + 0.60 * (employment == "Unemployed")
    + 0.30 * (employment == "Student")
    + rng.normal(0, 0.50, N)               # irreducible noise
)
prob_default = 1 / (1 + np.exp(-logit))
default = (rng.random(N) < prob_default).astype(int)

df = pd.DataFrame({
    "Customer_ID":           range(1, N + 1),
    "Age":                   age,
    "Income":                income,
    "Employment_Status":     employment,
    "Monthly_Spending":      spend,
    "BNPL_Transactions":     bnpl,
    "Avg_Transaction_Value": atv,
    "Late_Payments":         late,
    "Credit_Score":          credit,
    "Spending_to_Income":    sti.round(3),
    "Default":               default,
})
df.to_csv(os.path.join(OUT, "bnpl_synthetic_dataset.csv"), index=False)

print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} cols")
print("Default rate:", round(df['Default'].mean(), 3))
print(df['Default'].value_counts())


# ==========================================================================
# TABLE 3.1 — Dataset variables and descriptions
# ==========================================================================
pd.DataFrame([
    ("Customer_ID",           "Identifier",  "Unique customer reference (dropped before modelling)"),
    ("Age",                   "Numeric",     "Age of the customer in years (18-59)"),
    ("Income",                "Numeric",     "Annual income of the customer"),
    ("Employment_Status",     "Categorical", "Student, Salaried, Self-employed or Unemployed"),
    ("Monthly_Spending",      "Numeric",     "Average monthly spending"),
    ("BNPL_Transactions",     "Numeric",     "Number of BNPL transactions made"),
    ("Avg_Transaction_Value", "Numeric",     "Average value of BNPL transactions"),
    ("Late_Payments",         "Numeric",     "Count of late repayments (0-6)"),
    ("Credit_Score",          "Numeric",     "Customer credit score (300-850)"),
    ("Spending_to_Income",    "Numeric",     "Engineered ratio of monthly spending to income"),
    ("Default",               "Target",      "1 = customer defaulted; 0 = repaid on time"),
], columns=["Variable", "Type", "Description"]
).to_csv(os.path.join(OUT, "table_3_1_dataset_features.csv"), index=False)


# ==========================================================================
# TABLE 3.2 — Target encoding   |   STEP 2: data check
# ==========================================================================
pd.DataFrame([("Repaid on time", 0, "Non-default (negative class)"),
              ("Defaulted", 1, "Default (positive class)")],
             columns=["Outcome", "Default", "Interpretation"]
            ).to_csv(os.path.join(OUT, "table_3_2_target_encoding.csv"), index=False)

print("\nMissing values:\n", df.isnull().sum())
print("\nDescriptive statistics:\n", df.describe().round(2))


# ==========================================================================
# STEP 3: FEATURE / TARGET SPLIT + COLUMN TYPES
# ==========================================================================
X = df.drop(columns=["Customer_ID", "Default"])
y = df["Default"]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
print("\nNumeric   :", numeric_features)
print("Categorical:", categorical_features)


# ==========================================================================
# FIGURE 4.1 — Distribution of the target variable
# ==========================================================================
plt.figure(figsize=(6, 4.5))
counts = df["Default"].value_counts().sort_index()
ax = sns.barplot(x=counts.index, y=counts.values,
                 hue=counts.index, palette=["#2a8a5b", "#d35450"], legend=False)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Repaid (0)", "Default (1)"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}\n({v/len(df):.1%})", ha="center", va="bottom")
plt.title("Distribution of the Target Variable (Default)")
plt.ylabel("Number of customers"); plt.xlabel("")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_4_1_target_distribution.png"), dpi=150)
plt.show()

# ==========================================================================
# FIGURE 4.2 — Histograms of numeric features
# ==========================================================================
df[numeric_features].hist(figsize=(14, 8), bins=20, color="#3b6ea5", edgecolor="white")
plt.suptitle("Histograms of Numeric Features", y=1.02, fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_4_2_histograms.png"), dpi=150, bbox_inches="tight")
plt.show()

# ==========================================================================
# FIGURE 4.3 — Correlation heatmap (numeric features + target)
# ==========================================================================
plt.figure(figsize=(9, 7.5))
corr = df[numeric_features + ["Default"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_4_3_correlation_heatmap.png"), dpi=150)
plt.show()


# ==========================================================================
# STEP 4: PREPROCESSING PIPELINE  (impute -> scale / one-hot)
# ==========================================================================
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

# ==========================================================================
# STEP 5: TRAIN-TEST SPLIT  (stratified)  +  TABLE 3.4
# ==========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
pd.DataFrame([("Training", "80%", len(X_train)),
              ("Testing", "20%", len(X_test)),
              ("Total", "100%", len(X))],
             columns=["Subset", "Proportion", "Records"]
            ).to_csv(os.path.join(OUT, "table_3_4_train_test_split.csv"), index=False)
print(f"\nTrain: {len(X_train)}   Test: {len(X_test)}")

# ==========================================================================
# TABLE 3.5 — Hyperparameters
# ==========================================================================
pd.DataFrame([
    ("Logistic Regression", "max_iter=1000; default L2 regularisation"),
    ("XGBoost", "n_estimators=200, max_depth=5, learning_rate=0.05, "
                "subsample=0.8, colsample_bytree=0.8, eval_metric='logloss'"),
], columns=["Model", "Key hyperparameters"]
).to_csv(os.path.join(OUT, "table_3_5_hyperparameters.csv"), index=False)


# ==========================================================================
# STEP 6: TRAIN BOTH MODELS
# ==========================================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=SEED),
}

fitted, preds, probas, rows = {}, {}, {}, []
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    pipe.fit(X_train, y_train)
    p, pr = pipe.predict(X_test), pipe.predict_proba(X_test)[:, 1]
    fitted[name], preds[name], probas[name] = pipe, p, pr
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, p),
                 "Precision": precision_score(y_test, p),
                 "Recall": recall_score(y_test, p),
                 "F1": f1_score(y_test, p),
                 "ROC_AUC": roc_auc_score(y_test, pr)})
    print(f"\n===== {name} =====")
    print("Confusion matrix:\n", confusion_matrix(y_test, p))
    print(classification_report(y_test, p, digits=3))


def cm_df(name):
    cm = confusion_matrix(y_test, preds[name])
    return pd.DataFrame(cm,
        index=["Actual: No-default (0)", "Actual: Default (1)"],
        columns=["Predicted: No-default (0)", "Predicted: Default (1)"])

cm_df("Logistic Regression").to_csv(os.path.join(OUT, "table_4_1_logreg_confusion.csv"))
cm_df("XGBoost").to_csv(os.path.join(OUT, "table_4_2_xgb_confusion.csv"))

comparison = pd.DataFrame(rows).set_index("Model").round(3)
comparison.to_csv(os.path.join(OUT, "table_4_3_model_comparison.csv"))
print("\n===== Model Comparison =====\n", comparison)


# ==========================================================================
# FIGURE 4.4 — ROC curves (Logistic Regression vs XGBoost)
# ==========================================================================
fig, ax = plt.subplots(figsize=(7, 6))
for name in models:
    RocCurveDisplay.from_predictions(y_test, probas[name], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("ROC Curves — Logistic Regression vs XGBoost")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_4_4_roc_curves.png"), dpi=150)
plt.show()

# ==========================================================================
# FIGURE 4.5 — Confusion-matrix heatmaps
# ==========================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, models):
    sns.heatmap(confusion_matrix(y_test, preds[name]), annot=True, fmt="d",
                cmap="Blues", cbar=False, ax=ax,
                xticklabels=["No-default", "Default"],
                yticklabels=["No-default", "Default"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.suptitle("Confusion-Matrix Heatmaps", y=1.03, fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_4_5_confusion_heatmaps.png"), dpi=150, bbox_inches="tight")
plt.show()

# ==========================================================================
# FIGURE 4.6 — XGBoost feature importance
# ==========================================================================
ohe = fitted["XGBoost"].named_steps["prep"].named_transformers_["cat"].named_steps["encode"]
feat_names = numeric_features + list(ohe.get_feature_names_out(categorical_features))
imp = (pd.DataFrame({"Feature": feat_names,
                     "Importance": fitted["XGBoost"].named_steps["clf"].feature_importances_})
       .sort_values("Importance", ascending=False).reset_index(drop=True))
imp.to_csv(os.path.join(OUT, "table_4_4_feature_importance.csv"), index=False)
print("\n===== Feature Importance =====\n", imp.head(10).round(3))

top = imp.head(10)[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top["Feature"], top["Importance"], color="#2a8a5b")
plt.title("Feature Importance — XGBoost"); plt.xlabel("Importance")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_4_6_feature_importance.png"), dpi=150)
plt.show()


# ==========================================================================
# STEP 7: PREDICT RISK FOR A NEW CUSTOMER
# ==========================================================================
new_customer = pd.DataFrame([{
    "Age": 25, "Income": 30_000, "Employment_Status": "Student",
    "Monthly_Spending": 25_000, "BNPL_Transactions": 12,
    "Avg_Transaction_Value": 1_800, "Late_Payments": 3,
    "Credit_Score": 620, "Spending_to_Income": round(25_000 / 30_000, 3),
}])
best = fitted["Logistic Regression"]      # best model on this dataset
proba = best.predict_proba(new_customer)[0][1]
print("\n===== New Customer Prediction =====")
print(f"Default probability: {proba:.2%}  ->",
      "HIGH RISK" if proba >= 0.5 else "LOW RISK")

print("\nDone. Figures + tables saved to ./outputs")


ModuleNotFoundError: No module named 'numpy'